# Januário Leads — Piloto nacional Overture (27 UFs)

Este notebook usa o **mesmo importador que já funcionou em Lavras e Varginha**, copiado internamente na célula 2. Não depende de baixar código de repositório privado ou GitHub Actions.

**Objetivo:** verificar uma cidade representativa de cada UF, até **30 estabelecimentos por UF**, no Supabase **Januário Leads — Staging**. Isso **não é uma importação de todas as cidades** nem cobertura nacional completa. Minas Gerais já tem registros, portanto será pulado se Varginha tiver 30.

**Segurança:** destino fixo no projeto de homologação `prlrxjothrslhswpkoza`, limite de 100 MiB no importador e parada preventiva aos 95 MiB, chave `service_role` inserida apenas em campo oculto, sem salvar em arquivo. Não usa Google Places, Geoapify nem APIs pagas.

**Como executar:** rode as células 1, 2 e 3 uma vez; depois rode a 4, digite `IMPORTAR_BRASIL` e forneça a **mesma chave de homologação** que funcionou antes. O padrão é até 3 UFs novas por rodada; **repita só a célula 4** para continuar. Se a sessão desconectar, reabra o notebook e execute 1–4; cidades com 30 registros já gravados serão puladas automaticamente. Não mostre a chave a ninguém.

**Observação:** a duração e a disponibilidade do Google Colab gratuito variam. Cada estado baixa o recorte Overture e a malha IBGE do município. Se houver uma falha, a execução para e mantém o que já foi gravado. A situação do estado deve permanecer `partial`.


In [ ]:
# Célula 1 — instalar ferramentas necessárias
%pip install -q "overturemaps>=2.1,<3" "geopandas>=1,<2" "shapely>=2,<3" "pyogrio>=0.10,<1"
print("Dependências instaladas.")

In [ ]:
# Célula 2 — disponibilizar o importador localmente, sem acessar o GitHub privado.
# Cópia do script do Januário Leads; não contém chave ou segredo.
from pathlib import Path
_IMPORTER = (
    "#!/usr/bin/env python3\n\"\"\"Import Overture Places into private Supabase table, scoped to Brazil's municipalitie"
    "s.\n\nPre-req:\n    pip install overturemaps geopandas shapely pyogrio\n    export SUPABASE_URL='https://<project>"
    ".supabase.co'\n    export SUPABASE_SERVICE_ROLE_KEY='<service role secret>'\n    python scripts/import_overture_"
    "br.py --states MG,SP --database-max-mb 400\n\nThe runner covers all 27 federal units by default. Each state is s"
    "treamed\nseparately; it does not attempt to download and retain a global Overture file.\nIt stops BEFORE reachin"
    "g a configured database threshold, and reports partial\ncoverage honestly. Never run from a browser or publish "
    "the service-role key.\n\"\"\"\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subp"
    "rocess\nimport sys\nimport tempfile\nimport time\nimport unicodedata\nfrom urllib import request, error\nfrom urllib"
    ".parse import urlparse\n\nIBGE_2024_ROOT = (\n    \"https://geoftp.ibge.gov.br/organizacao_do_territorio/\"\n    \"ma"
    "lhas_territoriais/malhas_municipais/municipio_2024\"\n)\n\ndef state_boundaries_url(state):\n    \"\"\"IBGE offers one"
    " ZIP per UF; do not download the 199 MB national file.\"\"\"\n    if state not in STATE_CODES:\n        raise Value"
    "Error(\"Invalid Brazilian state code\")\n    return f\"{IBGE_2024_ROOT}/UFs/{state}/{state}_Municipios_2024.zip\"\n\n"
    "def ensure_state_boundaries(state, source=None):\n    if source:\n        path = Path(source)\n        if not pat"
    "h.is_file():\n            raise RuntimeError(f\"IBGE boundaries file not found: {path}\")\n        return path\n   "
    " path = Path(\"data/ibge/\") / f\"{state}_Municipios_2024.zip\"\n    if not path.is_file():\n        path.parent.mkd"
    "ir(parents=True, exist_ok=True)\n        url = state_boundaries_url(state)\n        print(f\"[{state}] Downloadin"
    "g IBGE municipal boundaries: {url}\", flush=True)\n        temp = path.with_suffix(\".partial\")\n        try:\n    "
    "        request.urlretrieve(url, temp)\n            if temp.stat().st_size < 512:\n                raise Runtime"
    "Error(\"IBGE boundaries download appears empty\")\n            temp.rename(path)\n        finally:\n            tem"
    "p.unlink(missing_ok=True)\n    return path\nSTATE_CODES = (\n    \"AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR"
    " PE PI \"\n    \"RJ RN RS RO RR SC SP SE TO\"\n).split()\nBATCH_SIZE = 150\n\n\ndef normalize(value):\n    \"\"\"Match the "
    "TypeScript search adapter's city/name normalization exactly.\"\"\"\n    letters = \"\".join(\n        c for c in unic"
    "odedata.normalize(\"NFD\", value.casefold())\n        if unicodedata.category(c) != \"Mn\"\n    )\n    return \" \".joi"
    "n(re.sub(r\"[^a-z0-9]+\", \" \", letters).split())\n\n\ndef first(values):\n    return next((v for v in (values or [])"
    " if isinstance(v, str) and v.strip()), None)\n\n\n# A social profile, WhatsApp contact or link shortener is NOT a"
    " verified\n# business website. This preserves truthful \"sem site\" prospecting filters.\nNON_WEBSITE_HOSTS = froz"
    "enset((\n    \"wa.me\", \"whatsapp.com\", \"api.whatsapp.com\", \"instagram.com\",\n    \"facebook.com\", \"fb.me\", \"m.me\","
    " \"linktr.ee\", \"linktree.com\",\n    \"bit.ly\", \"tinyurl.com\", \"t.co\", \"youtube.com\", \"youtu.be\",\n    \"tiktok.com\""
    ", \"x.com\", \"twitter.com\", \"maps.google.com\",\n    \"goo.gl\", \"beacons.ai\",\n))\n\n\ndef is_business_website(value):\n"
    "    if not isinstance(value, str):\n        return False\n    try:\n        url = urlparse(value.strip())\n       "
    " host = (url.hostname or \"\").lower().removeprefix(\"www.\")\n        return (url.scheme in (\"http\", \"https\")\n    "
    "            and bool(host) and \".\" in host\n                and host not in NON_WEBSITE_HOSTS\n                a"
    "nd not any(host.endswith(\".\" + blocked)\n                            for blocked in NON_WEBSITE_HOSTS))\n    exc"
    "ept ValueError:\n        return False\n\n\ndef first_website(values):\n    return first([value for value in (values"
    " or []) if is_business_website(value)])\n\n\ndef select_diverse_pool(records, maximum):\n    \"\"\"Reserve 40% for co"
    "ntactable companies without identified websites.\n\n    Fill unused slots with the best available business conta"
    "cts. Never promise\n    that a missing website is proof a business actually has no website.\n    \"\"\"\n    ranked "
    "= sorted(records, key=lead_quality, reverse=True)\n    no_site = [r for r in ranked if not r.get(\"website\")\n   "
    "            and (r.get(\"phone\") or r.get(\"email\") or r.get(\"instagram\"))]\n    target_no_site = min(len(no_site"
    "), (maximum * 2 + 4) // 5)\n    chosen = no_site[:target_no_site]\n    selected = {r[\"id\"] for r in chosen}\n    "
    "for row in ranked:\n        if len(chosen) >= maximum:\n            break\n        if row[\"id\"] not in selected:\n"
    "            chosen.append(row)\n            selected.add(row[\"id\"])\n    return chosen\n\n\ndef api(base, key, path"
    ", method=\"GET\", payload=None):\n    data = json.dumps(payload, ensure_ascii=False).encode() if payload is not N"
    "one else None\n    headers = {\n        \"apikey\": key, \"Authorization\": f\"Bearer {key}\",\n        \"Content-Type\":"
    " \"application/json\",\n        \"Prefer\": \"resolution=merge-duplicates,return=minimal\",\n    }\n    for attempt in "
    "range(5):\n        try:\n            req = request.Request(base + path, method=method, headers=headers, data=dat"
    "a)\n            with request.urlopen(req, timeout=45) as resp:\n                body = resp.read()\n             "
    "   return json.loads(body) if body else None\n        except error.HTTPError as exc:\n            detail = exc.r"
    "ead(300).decode(\"utf-8\", errors=\"replace\")\n            if exc.code not in (429, 500, 502, 503, 504) or attempt"
    " == 4:\n                raise RuntimeError(f\"Supabase API {exc.code}: {detail}\") from exc\n        except (error"
    ".URLError, TimeoutError):\n            if attempt == 4:\n                raise\n        time.sleep(min(2 ** attem"
    "pt, 15))\n    raise RuntimeError(\"Unreachable API retry state\")\n\n\ndef database_bytes(base, key):\n    result = a"
    "pi(base, key, \"/rest/v1/rpc/overture_storage_bytes\", method=\"POST\", payload={})\n    if not isinstance(result, "
    "int):\n        raise RuntimeError(\"Storage guard unavailable: run SQL migration first.\")\n    return result\n\n\nde"
    "f status(base, key, state, phase, count=0, message=None):\n    api(base, key, \"/rest/v1/overture_import_progres"
    "s?on_conflict=state\", \"POST\", [{\n        \"state\": state, \"status\": phase,\n        \"imported_rows\": count, \"las"
    "t_error\": message,\n        \"checked_at\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime()),\n    }])\n\n\ndef mun"
    "icipalities_index(boundaries):\n    import geopandas as gpd\n\n    frame = gpd.read_file(\"zip://\" + str(boundarie"
    "s.resolve()))\n    if frame.crs is None:\n        raise RuntimeError(\"The IBGE municipal boundary file has no CR"
    "S.\")\n    frame = frame.to_crs(epsg=4326).reset_index(drop=True)\n    columns = set(frame.columns)\n    if not {\""
    "NM_MUN\", \"SIGLA_UF\"}.issubset(columns):\n        raise RuntimeError(f\"Unexpected IBGE municipalities columns: {"
    "sorted(columns)}\")\n    frame = frame[frame[\"SIGLA_UF\"].isin(STATE_CODES)].copy().reset_index(drop=True)\n    re"
    "turn frame\n\n\ndef place_record(feature, municipalities, state):\n    from shapely.geometry import Point\n\n    pro"
    "ps = feature.get(\"properties\") or {}\n    coords = (feature.get(\"geometry\") or {}).get(\"coordinates\") or []\n   "
    " if len(coords) != 2:\n        return None\n    lon, lat = coords\n    if not isinstance(lon, (int, float)) or no"
    "t isinstance(lat, (int, float)):\n        return None\n    name = (props.get(\"names\") or {}).get(\"primary\")\n    "
    "# GeoJSON exporters may place the Overture GERS id in Feature.id rather\n    # than properties.id. Preserve eit"
    "her representation.\n    identifier = props.get(\"id\") or feature.get(\"id\")\n    if not isinstance(name, str) or "
    "not name.strip() or not isinstance(identifier, str):\n        return None\n    if props.get(\"operating_status\") "
    "== \"permanently_closed\":\n        return None\n\n    # State bounding boxes overlap: map point to the true munici"
    "pality.\n    point = Point(lon, lat)\n    matches = municipalities.sindex.query(point, predicate=\"intersects\")\n "
    "   if len(matches) == 0:\n        return None\n    mun = municipalities.iloc[int(matches[0])]\n    if mun[\"SIGLA_"
    "UF\"] != state:\n        return None\n    address = (props.get(\"addresses\") or [{}])[0]\n    if not isinstance(add"
    "ress, dict):\n        address = {}\n    taxonomy = props.get(\"taxonomy\") or {}\n    confidence = props.get(\"confi"
    "dence\")\n    socials = props.get(\"socials\") or []\n    websites = props.get(\"websites\") or []\n    instagram = fi"
    "rst([s for s in (socials + websites)\n                       if isinstance(s, str) and\n                       r"
    "e.search(r\"https?://(?:www\\.)?instagram\\.com/\", s, re.I)])\n    return {\n        \"id\": identifier[:100],\n      "
    "  \"name\": name.strip()[:240],\n        \"name_key\": normalize(name)[:240],\n        \"category\": str(taxonomy.get("
    "\"primary\") or props.get(\"basic_category\") or \"\")[:120] or None,\n        \"city\": str(mun[\"NM_MUN\"])[:120],\n    "
    "    \"city_key\": normalize(str(mun[\"NM_MUN\"]))[:120],\n        \"state\": str(mun[\"SIGLA_UF\"]),\n        \"address\":"
    " str(address.get(\"freeform\") or \"\")[:400] or None,\n        \"latitude\": lat, \"longitude\": lon,\n        \"phone\":"
    " (first(props.get(\"phones\")) or \"\")[:80] or None,\n        \"email\": (first(props.get(\"emails\")) or \"\")[:200] or"
    " None,\n        \"website\": (first_website(websites) or \"\")[:500] or None,\n        \"instagram\": instagram[:500] "
    "if instagram else None,\n        \"confidence\": confidence if isinstance(confidence, (int, float)) else None,\n  "
    "      \"operating_status\": str(props.get(\"operating_status\") or \"\")[:35] or None,\n        \"data_release\": os.ge"
    "tenv(\"OVERTURE_DATA_RELEASE\", \"latest\")[:80],\n    }\n\n\ndef lead_quality(row):\n    \"\"\"Prioritize genuine, contac"
    "table establishments within the per-city quota.\"\"\"\n    confidence = row.get(\"confidence\")\n    weight = float(c"
    "onfidence) * 5 if isinstance(confidence, (int, float)) else 0\n    return (weight\n            + 12 * bool(row.g"
    "et(\"phone\"))\n            + 11 * bool(row.get(\"email\"))\n            + 7 * bool(row.get(\"website\"))\n            "
    "+ 3 * bool(row.get(\"instagram\"))\n            + 2 * bool(row.get(\"category\")))\n\n\ndef main():\n    parser = argpa"
    "rse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--states\", default=\",\".join(STATE_CODES),\n   "
    "                     help=\"Comma-separated UF codes; default all 27\")\n    parser.add_argument(\"--cities\", defa"
    "ult=\"\",\n                        help=\"Optional comma-separated cities, requires exactly one UF; useful for a s"
    "mall pilot\")\n    parser.add_argument(\"--boundaries\", default=None,\n                        help=\"Optional exis"
    "ting ZIP; defaults to one IBGE ZIP per UF\")\n    parser.add_argument(\"--database-max-mb\", type=int, default=400"
    ")\n    parser.add_argument(\"--max-records-per-state\", type=int, default=0,\n                        help=\"0=no s"
    "tate cap; nonzero limits state rows and marks coverage partial\")\n    parser.add_argument(\"--max-records-per-ci"
    "ty\", type=int, default=20,\n                        help=\"Per-municipality safety cap, default 20 (0=unlimited)"
    "\")\n    parser.add_argument(\"--dry-run\", action=\"store_true\")\n    args = parser.parse_args()\n    if args.databa"
    "se_max_mb < 30 or args.database_max_mb > 480:\n        parser.error(\"--database-max-mb must be between 30 and 4"
    "80 MB\")\n    if args.max_records_per_state < 0:\n        parser.error(\"--max-records-per-state cannot be negativ"
    "e\")\n    if not 1 <= args.max_records_per_city <= 100:\n        parser.error(\"--max-records-per-city must be bet"
    "ween 1 and 100\")\n    states = [s.strip().upper() for s in args.states.split(\",\")]\n    if not states or len(set"
    "(states)) != len(states) or any(s not in STATE_CODES for s in states):\n        parser.error(\"--states must con"
    "tain unique Brazilian UF codes\")\n    cities = [normalize(s.strip()) for s in args.cities.split(\",\") if s.strip"
    "()]\n    if cities and len(states) != 1:\n        parser.error(\"--cities requires exactly one UF in --states\")\n\n"
    "    base = os.getenv(\"SUPABASE_URL\", \"\").rstrip(\"/\")\n    key = os.getenv(\"SUPABASE_SERVICE_ROLE_KEY\", \"\")\n    "
    "if not args.dry_run and (not base.startswith(\"https://\") or not key):\n        parser.error(\"SUPABASE_URL and S"
    "UPABASE_SERVICE_ROLE_KEY are required\")\n\n    limit_bytes = args.database_max_mb * 1024 * 1024\n    for state in"
    " states:\n        # Individual state archive keeps network and runner-memory use modest.\n        boundaries = e"
    "nsure_state_boundaries(state, args.boundaries)\n        municipalities = municipalities_index(boundaries)\n     "
    "   state_munis = municipalities.loc[municipalities.SIGLA_UF == state].copy()\n        if cities:\n            st"
    "ate_munis = state_munis.loc[\n                state_munis.NM_MUN.map(lambda name: normalize(str(name)) in citie"
    "s)\n            ].copy()\n            found_cities = {normalize(str(name)) for name in state_munis.NM_MUN}\n     "
    "       if set(cities) != found_cities:\n                raise RuntimeError(f\"Municipality not found in {state}:"
    " {sorted(set(cities) - found_cities)}\")\n        if state_munis.empty:\n            raise RuntimeError(f\"IBGE ar"
    "chive does not cover {state}\")\n        west, south, east, north = state_munis.total_bounds\n        # Slightly "
    "wider rectangle avoids border rounding gaps.\n        bbox = \",\".join(f\"{v:.6f}\" for v in (west - .005, south -"
    " .005, east + .005, north + .005))\n        print(f\"[{state}] Overture bbox {bbox}\", flush=True)\n        if arg"
    "s.dry_run:\n            continue\n        storage_before = database_bytes(base, key)\n        if storage_before >"
    "= limit_bytes:\n            print(f\"STOP: database reached configured {args.database_max_mb} MB safety limit.\")"
    "\n            return 2\n        total = 0\n        status(base, key, state, \"running\")\n        try:\n            w"
    "ith tempfile.TemporaryDirectory(prefix=\"overture-br-\") as temp:\n                out = Path(temp) / f\"{state}.g"
    "eojsonseq\"\n                subprocess.run([\"overturemaps\", \"download\", f\"--bbox={bbox}\",\n                     "
    "           \"--type=place\", \"-f\", \"geojsonseq\", \"-o\", str(out)],\n                               check=True)\n   "
    "             # Store only a small, quality-ranked per-city selection in memory.\n                city_records ="
    " {}\n                city_capped = False\n                with out.open(\"r\", encoding=\"utf-8\") as src:\n         "
    "           for line in src:\n                        line = line.lstrip(\"\\x1e\").strip()\n                       "
    " if not line:\n                            continue\n                        row = place_record(json.loads(line)"
    ", state_munis, state)\n                        if row is None:\n                            continue\n           "
    "             # Keep two bounded quality pools per municipality.\n                        # Otherwise the top 30"
    " are almost always ALL \"with site\".\n                        groups = city_records.setdefault(row[\"city_key\"], "
    "{\n                            \"with_site\": [], \"without_site\": [],\n                        })\n                "
    "        pool = groups[\"with_site\" if row.get(\"website\") else \"without_site\"]\n                        if len(po"
    "ol) < args.max_records_per_city:\n                            pool.append(row)\n                        else:\n  "
    "                          city_capped = True\n                            worst = min(range(len(pool)),\n       "
    "                                 key=lambda idx: lead_quality(pool[idx]))\n                            if lead_"
    "quality(row) > lead_quality(pool[worst]):\n                                pool[worst] = row\n\n                #"
    " Interleave municipality samples for nationwide fair coverage.\n                ranked_cities = {\n             "
    "       city: select_diverse_pool(groups[\"with_site\"] + groups[\"without_site\"],\n                               "
    "               args.max_records_per_city)\n                    for city, groups in city_records.items()\n       "
    "         }\n                selected = []\n                for rank in range(args.max_records_per_city):\n       "
    "             for city in sorted(ranked_cities):\n                        pool = ranked_cities[city]\n           "
    "             if rank < len(pool):\n                            selected.append(pool[rank])\n\n                if "
    "not selected:\n                    raise RuntimeError(\"Overture returned no usable places for this state\")\n    "
    "            # A municipality-only pilot never marks the whole state as imported.\n                partial = cit"
    "y_capped or bool(cities)\n                if args.max_records_per_state and len(selected) > args.max_records_pe"
    "r_state:\n                    selected = selected[:args.max_records_per_state]\n                    partial = Tr"
    "ue\n                for start in range(0, len(selected), BATCH_SIZE):\n                    if database_bytes(bas"
    "e, key) >= limit_bytes:\n                        partial = True\n                        break\n                 "
    "   batch = selected[start:start + BATCH_SIZE]\n                    api(base, key, \"/rest/v1/overture_places?on_"
    "conflict=id\", \"POST\", batch)\n                    total += len(batch)\n                storage_after = database_"
    "bytes(base, key)\n                status(base, key, state, \"partial\" if partial else \"complete\", total)\n       "
    "         delta = storage_after - storage_before\n                bytes_per_imported_row = delta // total if tot"
    "al else 0\n                print(f\"[{state}] {total} imported; \"\n                      f\"database before={stora"
    "ge_before}, after={storage_after}, \"\n                      f\"delta={delta}, average={bytes_per_imported_row} B"
    "/row; \"\n                      f\"{'PARTIAL' if partial else 'COMPLETE'}\", flush=True)\n                if storag"
    "e_after >= limit_bytes:\n                    print(\"STOP: database safety threshold; remaining states NOT impor"
    "ted.\", flush=True)\n                    return 2\n        except Exception as exc:\n            status(base, key,"
    " state, \"error\", total, str(exc)[:200])\n            raise\n    print(\"Ingestion pass ended. Check overture_impo"
    "rt_progress for coverage.\", flush=True)\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n"
)
_SCRIPT = Path("/content/import_overture_br.py")
compile(_IMPORTER, str(_SCRIPT), "exec")  # valida a sintaxe antes de executar
_SCRIPT.write_text(_IMPORTER, encoding="utf-8")
print("Importador preparado. Nenhum dado enviado ao Supabase ainda.")

In [ ]:
# Célula 3 — plano do piloto nacional: 1 município por UF, até 30 empresas por UF.
# MG já tem 60 empresas (Lavras e Varginha); Varginha será ignorada se já tiver 30.
PILOTO_27_UFS = [
    ("MG", "Varginha"), ("SP", "Campinas"), ("RJ", "Niterói"),
    ("AC", "Rio Branco"), ("AL", "Maceió"), ("AP", "Macapá"),
    ("AM", "Manaus"), ("BA", "Salvador"), ("CE", "Fortaleza"),
    ("DF", "Brasília"), ("ES", "Vitória"), ("GO", "Goiânia"),
    ("MA", "São Luís"), ("MT", "Cuiabá"), ("MS", "Campo Grande"),
    ("PA", "Belém"), ("PB", "João Pessoa"), ("PR", "Curitiba"),
    ("PE", "Recife"), ("PI", "Teresina"), ("RN", "Natal"),
    ("RS", "Porto Alegre"), ("RO", "Porto Velho"), ("RR", "Boa Vista"),
    ("SC", "Florianópolis"), ("SE", "Aracaju"), ("TO", "Palmas"),
]

LIMITE_POR_UF = 30 #@param {type:"integer"}
UFS_POR_RODADA = 3 #@param {type:"integer"}

_UFS = set("AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI RJ RN RS RO RR SC SP SE TO".split())
assert len(PILOTO_27_UFS) == 27
assert set(uf for uf, _ in PILOTO_27_UFS) == _UFS
assert type(LIMITE_POR_UF) is int and 1 <= LIMITE_POR_UF <= 30
assert type(UFS_POR_RODADA) is int and 1 <= UFS_POR_RODADA <= 27

print("PILOTO NACIONAL PRONTO (nenhuma importação executada nesta célula)")
print("27 UFs planejadas, 1 cidade por UF, até", LIMITE_POR_UF, "empresas por UF.")
print("Máximo de", UFS_POR_RODADA, "novas UFs por rodada; retome repetindo a célula 4.")
print("Somente Supabase homologação; teto operacional 100 MiB; proteção adicional em 95 MiB.")
print("A amostra NÃO significa cobertura completa de todos os municípios brasileiros.")


In [ ]:
# Célula 4 — importar com confirmação, retomada e autenticação privada.
# Atenção: use SOMENTE a chave service_role da HOMOLOGAÇÃO no campo oculto.
from getpass import getpass
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json
import re
import subprocess
import sys
import unicodedata
import os

STAGING = "https://prlrxjothrslhswpkoza.supabase.co"
TETO_MIB = 100
AVISO_PARADA_MIB = 95  # Margem antes do teto real.
MB = 1024 * 1024

def _normalizar(valor):
    base = "".join(c for c in unicodedata.normalize("NFD", valor.casefold())
                   if unicodedata.category(c) != "Mn")
    return " ".join(re.sub(r"[^a-z0-9]+", " ", base).split())

def _pedido(chave, caminho, metodo="GET", dados=None):
    cabecalhos = {
        "apikey": chave, "Authorization": "Bearer " + chave,
        "Accept": "application/json",
        "Content-Type": "application/json",
        "Prefer": "count=exact",
    }
    req = Request(STAGING + caminho, headers=cabecalhos,
                  data=json.dumps(dados).encode("utf-8") if dados is not None else None,
                  method=metodo)
    with urlopen(req, timeout=45) as resposta:
        return resposta.read(), resposta.headers

def _tamanho_banco(chave):
    corpo, _ = _pedido(chave, "/rest/v1/rpc/overture_storage_bytes", "POST", {})
    valor = json.loads(corpo)
    if not isinstance(valor, int) or valor <= 0:
        raise RuntimeError("O banco não retornou um tamanho válido.")
    return valor

def _contagem_cidade(chave, uf, cidade):
    query = urlencode({
        "select": "id",
        "state": "eq." + uf,
        "city_key": "eq." + _normalizar(cidade),
    })
    _, headers = _pedido(chave, "/rest/v1/overture_places?" + query, "HEAD")
    faixa = headers.get("Content-Range", "")
    final = faixa.rsplit("/", 1)[-1]
    if not final.isdigit():
        raise RuntimeError("Não foi possível conferir o total de " + cidade + "/" + uf)
    return int(final)

def executar_piloto_nacional():
    confirmar = input("Para IMPORTAR de forma gradual nas 27 UFs (somente STAGING), digite IMPORTAR_BRASIL: ").strip().upper()
    if confirmar != "IMPORTAR_BRASIL":
        print("Cancelado: nenhuma empresa foi importada.")
        return

    chave = getpass("Chave service_role da HOMOLOGAÇÃO (campo oculto): ").strip()
    if not chave:
        raise ValueError("Chave vazia: execução cancelada.")
    try:
        # Falha antes de importar se a chave não tiver permissões apropriadas.
        tamanho = _tamanho_banco(chave)
        print("Homologação confirmada. Banco:", round(tamanho / MB, 2), "MiB.")
        if tamanho >= AVISO_PARADA_MIB * MB:
            print("PARADA DE SEGURANÇA: banco próximo do limite.")
            return

        importadas_nesta_rodada = 0
        concluidas = 0
        for uf, cidade in PILOTO_27_UFS:
            atual = _contagem_cidade(chave, uf, cidade)
            if atual >= LIMITE_POR_UF:
                print("JÁ EXISTE:", cidade, "/", uf, "-", atual, "empresas. Pulando.")
                concluidas += 1
                continue

            if importadas_nesta_rodada >= UFS_POR_RODADA:
                print("RODADA CONCLUÍDA. Execute a célula 4 novamente para continuar.")
                break

            tamanho = _tamanho_banco(chave)
            if tamanho >= AVISO_PARADA_MIB * MB:
                print("PARADA DE SEGURANÇA: limite preventivo de 95 MiB atingido.")
                break

            print("INICIANDO:", cidade, "/", uf, "- até", LIMITE_POR_UF, "estabelecimentos.")
            # Subprocesso recebe o segredo por variável de ambiente, nunca na linha de comando.
            ambiente = dict(os.environ)
            ambiente["SUPABASE_URL"] = STAGING
            ambiente["SUPABASE_SERVICE_ROLE_KEY"] = chave
            cmd = [
                sys.executable, str(_SCRIPT), "--states", uf, "--cities", cidade,
                "--max-records-per-city", str(LIMITE_POR_UF),
                "--max-records-per-state", str(LIMITE_POR_UF),
                "--database-max-mb", str(TETO_MIB),
            ]
            try:
                subprocess.run(cmd, check=True, timeout=1500, env=ambiente)
            except subprocess.TimeoutExpired:
                print("FALHA: tempo limite excedido em", cidade, "/", uf,
                      "- confira o banco antes de retomar.")
                break
            except subprocess.CalledProcessError as e:
                print("FALHA: importador interrompido em", cidade, "/", uf,
                      "- código", e.returncode, ". Sem avançar para outra UF.")
                break
            finally:
                ambiente.pop("SUPABASE_SERVICE_ROLE_KEY", None)
                del ambiente

            depois = _contagem_cidade(chave, uf, cidade)
            print("CONFERIDO NO SUPABASE:", cidade, "/", uf,
                  "| antes:", atual, "| agora:", depois)
            if depois < atual:
                raise RuntimeError("Contagem diminuiu inesperadamente: interrompido.")
            if depois == atual and atual == 0:
                print("FALHA: nenhum registro confirmado. Parando.")
                break
            importadas_nesta_rodada += 1
            concluidas += 1
        else:
            print("PLANO PERCORRIDO: todas as 27 UFs tiveram sua cidade verificada.")

        restante = 27 - concluidas
        print("Rodada: ", importadas_nesta_rodada,
              "novas UFs processadas | faltam verificar:", restante)
        print("Consulta final no Supabase: overture_import_progress e overture_import_city_progress.")
        print("ATENÇÃO: cobertura dos 27 estados é piloto; não significa todas as cidades/empresas.")
    finally:
        chave = ""  # Não persiste chave em variáveis de ambiente nem em arquivo.
        print("Chave encerrada nesta chamada. Desconecte o Colab ao terminar.")

executar_piloto_nacional()


## Depois do piloto

Envie somente as mensagens de resultado (sem chave). Conferiremos os totais por UF e cidade diretamente no Supabase.

Para avaliar cobertura completa dos municípios brasileiros, é necessária uma segunda fase com lotes distribuídos e controle de armazenamento. **Não libere produção** apenas por concluir este piloto.
